# Feature Engineering Pipeline
### Commodities Futures: Energy and Metals

This notebook walks through the full feature engineering pipeline for a cross-sectional futures dataset covering energy (CL, HO, RB, NG) and metals (GC, SI, HG, PL) contracts.

The pipeline has four tiers:
- **Tier 1 (Base Features):** Returns, volatility estimators, volume/OI, technical indicators, drawdown, and trend-scanning t-statistics
- **Tier 2 (Domain Features):** Asset-class-specific cross-sectional signals (energy crack spreads, metals ratios)
- **Tier 3 (Latent Features):** Walk-forward GMM, HMM, and K-means regime models
- **Tier 4 (Primary Signal Features):** Lags, streaks, and agreement statistics on the primary model signal

All rolling and model-based features are strictly backward-looking. No future data enters any feature at any point.

---
## 0. Configuration

All parameters (window lengths, model hyperparameters, file paths) live in a single `FeatureConfig` dataclass. Changing a parameter here propagates through the entire pipeline. No magic numbers are scattered through the code.

In [2]:
from dataclasses import dataclass, field


ENERGY: list[str] = ["cl1s", "ho1s", "rb1s", "ng1s"]
METALS: list[str] = ["gc1s", "si1s", "hg1s", "pl1s"]
PRECIOUS: list[str] = ["gc1s", "si1s", "pl1s"]


@dataclass
class FeatureConfig:
    """
    Central configuration for all feature engineering parameters.
    Pass a modified instance to override defaults in experiments.
    """

    # File paths
    ohlcv_path: str = "data/ohlcv_data.csv"
    signals_path: str = "data/primary_signals.csv"
    output_dir: str = "features"

    # Universe
    energy_instruments: list[str] = field(default_factory=lambda: list(ENERGY))
    metals_instruments: list[str] = field(default_factory=lambda: list(METALS))
    precious_metals: list[str] = field(default_factory=lambda: list(PRECIOUS))

    # Rolling windows (in trading days)
    vol_short: int = 5
    vol_mid: int = 20
    vol_long: int = 60
    rsi_period: int = 14
    macd_fast: int = 12
    macd_slow: int = 26
    macd_signal: int = 9
    bb_period: int = 20
    atr_period: int = 14
    adx_period: int = 14
    drawdown_window: int = 60
    oi_z_window: int = 60
    volume_z_window: int = 20
    corr_window: int = 60

    # Trend scanning horizons
    trend_horizons: list[int] = field(default_factory=lambda: [10, 20, 60])

    # Latent model parameters
    gmm_n_components: int = 3       # low / mid / high vol regimes
    hmm_n_states: int = 2           # calm / turbulent
    kmeans_k: int = 3
    refit_frequency: int = 63       # roughly quarterly (trading days)
    warmup: int = 504               # roughly 2 years before first refit

    # Misc
    random_state: int = 42
    signal_lags: list[int] = field(default_factory=lambda: [1, 5])
    signal_agreement_window: int = 5

---
## 1. Data Loading

We load two files:
- **OHLCV data:** Daily open, high, low, close, volume, and open interest for each futures contract. Long format with one row per (date, instrument). Data goes back to 1990.
- **Primary signals:** A wide-format CSV with one column per instrument containing the primary model signal (-1, 0, or +1). Starts from 2020. This is melted to long format for consistency.

The OHLCV is filtered to the configured energy and metals universe, then sorted by (instrument, date).

In [27]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

REQUIRED_OHLCV = {"date", "instrument", "open", "high", "low", "close", "volume", "open_interest"}


def load_ohlcv(cfg: FeatureConfig) -> pd.DataFrame:
    """
    Load and validate the OHLCV CSV.

    Returns a long-format DataFrame sorted by (instrument, date).
    Only rows for the configured energy and metals universe are kept.
    """
    
    df = pd.read_csv('../data/src/ohlcv_data.csv', parse_dates=["date"])
    missing = REQUIRED_OHLCV - set(df.columns)
    if missing:
        raise ValueError(f"OHLCV missing columns: {missing}")

    universe = set(cfg.energy_instruments) | set(cfg.metals_instruments)
    df = df[df["instrument"].isin(universe)].copy()

    for col in ["open", "high", "low", "close", "volume", "open_interest"]:
        df[col] = df[col].astype("float64")

    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    logger.info(
        "Loaded OHLCV: %d rows, %d instruments, %s to %s",
        len(df), df["instrument"].nunique(),
        df["date"].min().date(), df["date"].max().date(),
    )
    return df


def load_signals(cfg: FeatureConfig) -> pd.DataFrame:
    """
    Load primary signals CSV and melt to long format.

    The primary signal is in {-1, 0, +1}. It is used as a feature input
    to the metamodel, not as the label itself.
    """

    sig = pd.read_csv('../data/src/primary_signals.csv', parse_dates=["date"])
    universe = set(cfg.energy_instruments) | set(cfg.metals_instruments)
    ticker_cols = [c for c in sig.columns if c != "date" and c.lower() in universe]

    sig_long = sig.melt(
        id_vars="date",
        value_vars=ticker_cols,
        var_name="instrument",
        value_name="primary_signal",
    )
    sig_long["instrument"] = sig_long["instrument"].str.lower()
    sig_long["primary_signal"] = sig_long["primary_signal"].astype("int8")
    sig_long = sig_long.sort_values(["instrument", "date"]).reset_index(drop=True)

    return sig_long


def save_features(features: pd.DataFrame, dictionary: pd.DataFrame, cfg: FeatureConfig) -> None:
    """Persist the feature matrix as Parquet and the feature dictionary as CSV."""
    out = Path(cfg.output_dir)
    out.mkdir(parents=True, exist_ok=True)

    parquet_path = out / "features.parquet"
    dict_path = out / "feature_dictionary.csv"

    features.to_parquet(parquet_path, index=False)
    dictionary.to_csv(dict_path, index=False)

    logger.info("Saved features to %s (%d rows x %d cols)", parquet_path, len(features), len(features.columns))
    logger.info("Saved feature dictionary to %s (%d features)", dict_path, len(dictionary))

---
## 2. Tier 1: Base Features

Base features are computed per instrument independently. Each instrument's time series is sorted by date, the feature functions are applied, and the results are concatenated back into long format.

The sections below cover:
1. Returns (pct and log)
2. Volatility estimators (close-to-close, Parkinson, Garman-Klass)
3. Volume and open interest features
4. Technical indicators (RSI, MACD, Bollinger Bands, ATR, ADX)
5. Rolling drawdown and run-up
6. Trend-scanning t-statistics (Lecture 1)

In [4]:
def _per_instrument(df: pd.DataFrame, fn) -> pd.DataFrame:
    """Apply fn to each instrument group (sorted by date) and concatenate."""
    out = []
    for inst, g in df.groupby("instrument", sort=False):
        g = g.sort_values("date").reset_index(drop=True)
        out.append(fn(g))
    return pd.concat(out, ignore_index=True)

### 2.1 Returns

We compute both percentage and log returns. Log returns are preferred throughout the pipeline because they are additive across time horizons and approximately normally distributed, which is required by the GMM and HMM emission models downstream.

In [5]:
def _add_returns(g: pd.DataFrame) -> pd.DataFrame:
    """
    Compute log and percentage returns for a single instrument.

    Features produced:
        ret_1d     : one-day pct return
        logret_1d  : one-day log return
        ret_5d     : rolling sum of logret_1d over 5 days
        ret_10d    : rolling sum over 10 days
        ret_20d    : rolling sum over 20 days
        ret_60d    : rolling sum over 60 days
    """
    c = g["close"]
    g = g.copy()
    g["ret_1d"] = c.pct_change()
    g["logret_1d"] = np.log(c / c.shift(1))
    for h in [5, 10, 20, 60]:
        g[f"ret_{h}d"] = g["logret_1d"].rolling(h).sum()
    return g

### 2.2 Volatility Estimators

We use three estimators:

- **Close-to-close (CC):** Rolling standard deviation of log returns. Simple but noisy.
- **Parkinson:** Uses only the high-low range. More efficient than CC for assets with large intraday moves. Formula: $\sigma^2 = \frac{1}{4\ln 2 \cdot N} \sum (\ln H/L)^2$
- **Garman-Klass:** Uses open, high, low, and close. Formula: $\sigma^2 = 0.5(\ln H/L)^2 - (2\ln 2 - 1)(\ln C/O)^2$

All windows are backward-only. No `center=True`.

In [6]:
def _add_volatility(g: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    """
    Rolling volatility estimators.

    Features produced:
        vol_20d         : 20-day CC vol
        vol_60d         : 60-day CC vol
        parkinson_20d   : 20-day Parkinson estimator
        garman_klass_20d: 20-day Garman-Klass estimator
        vol_of_vol_20d  : rolling std of vol_20d (regime transition signal)
        vol_ratio_5_60  : short/long vol ratio (>1 means vol is expanding)
    """
    g = g.copy()
    lr = g["logret_1d"]

    g["vol_20d"] = lr.rolling(cfg.vol_mid).std()
    g["vol_60d"] = lr.rolling(cfg.vol_long).std()

    # Parkinson: uses only high and low
    hl2 = np.log(g["high"] / g["low"]) ** 2
    g["parkinson_20d"] = np.sqrt(hl2.rolling(cfg.vol_mid).mean() / (4 * np.log(2)))

    # Garman-Klass: adds open and close
    hl_sq = 0.5 * np.log(g["high"] / g["low"]) ** 2
    co_sq = (2 * np.log(2) - 1) * np.log(g["close"] / g["open"]) ** 2
    gk_daily = hl_sq - co_sq
    g["garman_klass_20d"] = np.sqrt(gk_daily.rolling(cfg.vol_mid).mean())

    g["vol_of_vol_20d"] = g["vol_20d"].rolling(cfg.vol_mid).std()

    vol_5d = lr.rolling(cfg.vol_short).std()
    g["vol_ratio_5_60"] = vol_5d / g["vol_60d"]

    return g

### 2.3 Volume and Open Interest

Volume z-scores normalise for the secular growth in contract volume over decades. Open interest changes capture positioning build-up and unwind. Spikes in volume/OI ratio often precede roll dates or regime shifts.

In [7]:
def _add_volume_oi(g: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    """
    Volume and open-interest derived features.

    Features produced:
        volume_z_20d   : z-score of volume over 20-day rolling window
        oi_z_60d       : z-score of open interest over 60-day rolling window
        oi_change_5d   : 5-day pct change in open interest
        volume_oi_ratio: volume / open interest (turnover proxy)
    """
    g = g.copy()
    v = g["volume"]
    oi = g["open_interest"]

    vol_mean = v.rolling(cfg.volume_z_window).mean()
    vol_std = v.rolling(cfg.volume_z_window).std()
    g["volume_z_20d"] = (v - vol_mean) / vol_std.replace(0, np.nan)

    oi_mean = oi.rolling(cfg.oi_z_window).mean()
    oi_std = oi.rolling(cfg.oi_z_window).std()
    g["oi_z_60d"] = (oi - oi_mean) / oi_std.replace(0, np.nan)

    g["oi_change_5d"] = oi.pct_change(5)
    g["volume_oi_ratio"] = v / oi.replace(0, np.nan)

    return g

### 2.4 Technical Indicators

**RSI (14):** Wilder's Relative Strength Index. Measures momentum on a 0-100 scale. Above 70 is overbought, below 30 is oversold.

**MACD (12/26/9):** Difference of two EMAs. The histogram (MACD line minus signal line) captures momentum acceleration.

**Bollinger Bands (20):** Width measures regime volatility. Position (%B) captures where price sits within the band.

**ATR (14):** Average True Range. A measure of daily price range. Used downstream in Part 2 for dynamic barrier sizing in the triple-barrier labeling scheme.

**ADX (14):** Trend strength indicator. ADX above 25 signals a trending regime, which the metamodel can use to switch between momentum and mean-reversion strategies.

In [8]:
def _rsi(series: pd.Series, period: int) -> pd.Series:
    """Wilder RSI. Uses EWM with com=period-1 to match the original smoothing convention."""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - 100 / (1 + rs)


def _add_adx(g: pd.DataFrame, period: int) -> pd.DataFrame:
    """
    Wilder's ADX.

    Measures trend strength but not direction. Computed from the directional
    movement (DM+, DM-) and true range.

    Features produced:
        adx_14: 14-day ADX (0-100 scale; >25 indicates trending regime)
    """
    high = g["high"]
    low = g["low"]
    close = g["close"]

    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low - close.shift(1)).abs(),
    ], axis=1).max(axis=1)

    dm_pos = high.diff().clip(lower=0)
    dm_neg = (-low.diff()).clip(lower=0)
    dm_pos = dm_pos.where(dm_pos > dm_neg, 0)
    dm_neg = dm_neg.where(dm_neg > dm_pos, 0)

    atr = tr.ewm(com=period - 1, min_periods=period).mean()
    di_pos = 100 * dm_pos.ewm(com=period - 1, min_periods=period).mean() / atr.replace(0, np.nan)
    di_neg = 100 * dm_neg.ewm(com=period - 1, min_periods=period).mean() / atr.replace(0, np.nan)
    dx = 100 * (di_pos - di_neg).abs() / (di_pos + di_neg).replace(0, np.nan)

    g = g.copy()
    g["adx_14"] = dx.ewm(com=period - 1, min_periods=period).mean()
    return g


def _add_ta(g: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    """
    Technical indicators: RSI, MACD, Bollinger Bands, ATR, ADX.

    Features produced:
        rsi_14       : Wilder RSI (14)
        macd         : EMA(12) - EMA(26)
        macd_signal  : EMA of macd with span 9
        macd_hist    : macd - macd_signal
        bb_width_20  : (upper - lower) / mid (squeeze indicator)
        bb_pos_20    : (close - lower) / (upper - lower) (position within band)
        atr_14       : 14-day Average True Range
        adx_14       : 14-day Average Directional Index
    """
    g = g.copy()
    c = g["close"]

    g["rsi_14"] = _rsi(c, cfg.rsi_period)

    ema_fast = c.ewm(span=cfg.macd_fast, adjust=False).mean()
    ema_slow = c.ewm(span=cfg.macd_slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=cfg.macd_signal, adjust=False).mean()
    g["macd"] = macd_line
    g["macd_signal"] = signal_line
    g["macd_hist"] = macd_line - signal_line

    bb_mid = c.rolling(cfg.bb_period).mean()
    bb_std = c.rolling(cfg.bb_period).std()
    bb_upper = bb_mid + 2 * bb_std
    bb_lower = bb_mid - 2 * bb_std
    g["bb_width_20"] = (bb_upper - bb_lower) / bb_mid.replace(0, np.nan)
    g["bb_pos_20"] = (c - bb_lower) / (bb_upper - bb_lower).replace(0, np.nan)

    high_low = g["high"] - g["low"]
    high_prev = (g["high"] - g["close"].shift(1)).abs()
    low_prev = (g["low"] - g["close"].shift(1)).abs()
    tr = pd.concat([high_low, high_prev, low_prev], axis=1).max(axis=1)
    g["atr_14"] = tr.ewm(com=cfg.atr_period - 1, min_periods=cfg.atr_period).mean()

    g = _add_adx(g, cfg.adx_period)

    return g

### 2.5 Rolling Drawdown and Run-up

Drawdown from the 60-day high tells us how far we are below the recent peak, which conditions tail-risk models. `days_since_60d_high` measures how stale the last high is: a long lag in a downtrend signals a weakening move.

In [9]:
def _add_drawdown(g: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    """
    Rolling drawdown and run-up from 60-day extremes.

    Features produced:
        dd_from_60d_high    : close / rolling_max(60) - 1 (always <= 0)
        runup_from_60d_low  : close / rolling_min(60) - 1 (always >= 0)
        days_since_60d_high : bars since the rolling-60-day argmax
    """
    g = g.copy()
    c = g["close"]
    w = cfg.drawdown_window

    roll_max = c.rolling(w).max()
    roll_min = c.rolling(w).min()
    g["dd_from_60d_high"] = c / roll_max - 1
    g["runup_from_60d_low"] = c / roll_min - 1

    def _days_since_max(x: np.ndarray) -> float:
        if np.all(np.isnan(x)):
            return np.nan
        return float(len(x) - 1 - np.nanargmax(x))

    g["days_since_60d_high"] = (
        c.rolling(w, min_periods=w)
         .apply(_days_since_max, raw=True)
    )
    return g

### 2.6 Trend Scanning (Lecture 1)

For a lookback horizon H, we regress the closing price series over the most recent H bars on a time index (0, 1, ..., H-1) using OLS. The t-statistic on the slope coefficient is our trend strength measure.

$$\beta_1 = \frac{\sum (h_i - \bar{h})(y_i - \bar{y})}{\sum (h_i - \bar{h})^2} \qquad t = \frac{\beta_1}{\hat{\sigma}_{\beta_1}}$$

We compute this for H in {10, 20, 60} and also identify the horizon H* that maximises |t| (the max-|t| selection from Lecture 1). A large |t| at H=60 implies a sustained trend; at H=10 it implies a short burst. All regressions are backward-looking only.

In [10]:
def _trend_tstat_vectorised(close: pd.Series, H: int) -> pd.Series:
    """
    Backward-looking trend t-statistic for a fixed horizon H.

    At each time t, regresses close[t-H+1 : t+1] on h = 0..H-1 (OLS).
    Returns the t-statistic of the slope. NaN for the first H-1 observations.

    The h vector and Sxx = sum((h - h_mean)^2) are constant across all
    windows so we precompute them once.
    """
    h = np.arange(H, dtype=float)
    h_mean = h.mean()
    Sxx = float(((h - h_mean) ** 2).sum())

    y_vals = close.values.astype(float)
    N = len(y_vals)
    t_stats = np.full(N, np.nan)

    for t in range(H - 1, N):
        y = y_vals[t - H + 1 : t + 1]
        if np.any(np.isnan(y)):
            continue
        y_mean = y.mean()
        Sxy = float(((h - h_mean) * (y - y_mean)).sum())
        beta1 = Sxy / Sxx
        y_hat = (h - h_mean) * beta1 + y_mean
        resid = y - y_hat
        if H <= 2:
            t_stats[t] = np.nan
            continue
        mse = float((resid ** 2).sum()) / (H - 2)
        se = np.sqrt(mse / Sxx) if mse > 0 else np.nan
        t_stats[t] = beta1 / se if se and not np.isnan(se) else np.nan

    return pd.Series(t_stats, index=close.index)


def _add_trend_scanning(g: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    """
    Add trend-scanning t-statistics for each configured horizon.

    Features produced:
        trend_tstat_10  : t-stat for H=10 (short-term trend)
        trend_tstat_20  : t-stat for H=20 (medium-term trend)
        trend_tstat_60  : t-stat for H=60 (long-term trend)
        trend_tstat_best: t-stat at the horizon that maximises |t|
        trend_H_best    : the horizon H that achieved the maximum |t|
    """
    g = g.copy()
    c = g["close"]

    tstat_cols: dict[str, pd.Series] = {}
    for H in cfg.trend_horizons:
        col = f"trend_tstat_{H}"
        g[col] = _trend_tstat_vectorised(c, H)
        tstat_cols[col] = g[col]

    tstat_df = pd.DataFrame(tstat_cols, index=g.index)
    abs_tstat = tstat_df.abs()
    best_col_idx = abs_tstat.values.argmax(axis=1)

    horizons = np.array(cfg.trend_horizons)
    g["trend_tstat_best"] = tstat_df.values[np.arange(len(tstat_df)), best_col_idx]
    g["trend_H_best"] = horizons[best_col_idx].astype(float)

    all_nan_mask = abs_tstat.isna().all(axis=1)
    g.loc[all_nan_mask, ["trend_tstat_best", "trend_H_best"]] = np.nan

    return g

### 2.7 Base Feature Orchestrator

All per-instrument base feature functions are chained together and applied to each instrument in sequence.

In [11]:
def compute_base_features(ohlcv: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    """
    Compute all Tier 1 base features for every instrument.

    Takes the full long-format OHLCV DataFrame and returns it with all
    base feature columns appended. Date and instrument columns are preserved.
    """
    logger.info("Computing base features...")

    def _all_base(g: pd.DataFrame) -> pd.DataFrame:
        g = _add_returns(g)
        g = _add_volatility(g, cfg)
        g = _add_volume_oi(g, cfg)
        g = _add_ta(g, cfg)
        g = _add_drawdown(g, cfg)
        g = _add_trend_scanning(g, cfg)
        return g

    result = _per_instrument(ohlcv, _all_base)
    logger.info("Base features done: %d rows, %d columns", len(result), len(result.columns))
    return result

---
## 3. Tier 2a: Energy-Specific Features

Energy features are cross-sectional: they capture relationships between the four energy contracts (CL, HO, RB, NG) rather than properties of individual instruments.

Key features:

- **Energy basket return/vol:** Cross-sectional mean of 5-day cumulative returns and 20-day vol across all energy instruments.
- **3-2-1 crack spread proxy:** The refinery margin: 2 barrels of gasoline + 1 barrel of heating oil minus 3 barrels of crude. Computed in log-return space for scale invariance.
- **HO/CL and RB/CL log-price spreads:** Direct spread relationships between refined products and crude.
- **Relative return vs. basket:** How much each instrument deviates from the basket mean (idiosyncratic component).
- **Cyclical date encoding:** Month and day-of-week encoded as sine/cosine pairs. Season flags (winter heating demand, summer driving season).

In [12]:
def _pivot_close(df: pd.DataFrame, instruments: list[str]) -> pd.DataFrame:
    """Wide-format close prices: dates as index, instruments as columns."""
    sub = df[df["instrument"].isin(instruments)][["date", "instrument", "close"]].copy()
    return sub.pivot(index="date", columns="instrument", values="close")


def _pivot_logret(base_df: pd.DataFrame, instruments: list[str]) -> pd.DataFrame:
    """Wide-format logret_1d: dates as index, instruments as columns."""
    sub = base_df[base_df["instrument"].isin(instruments)][
        ["date", "instrument", "logret_1d"]
    ].copy()
    return sub.pivot(index="date", columns="instrument", values="logret_1d")


def compute_energy_features(base_df: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    """
    Compute all Tier 2a energy-specific features.

    Cross-sectional features are computed in wide format and then merged
    back to long format. All rolling windows are backward-looking.

    Returns a long-format DataFrame with only energy instrument rows.
    """
    energy = cfg.energy_instruments
    available = [i for i in energy if i in base_df["instrument"].unique()]
    if not available:
        raise ValueError("No energy instruments found in base_df.")

    lr_wide = _pivot_logret(base_df, available)
    close_wide = _pivot_close(base_df, available)

    # Energy basket: mean log return and mean vol across all energy contracts
    basket_ret5 = lr_wide.rolling(5).sum().mean(axis=1)
    basket_vol20 = lr_wide.rolling(20).std().mean(axis=1)
    basket_ret5.name = "energy_basket_ret_5d"
    basket_vol20.name = "energy_basket_vol_20d"
    basket = pd.concat([basket_ret5, basket_vol20], axis=1)

    # 3-2-1 crack spread: (2*RB + HO) / 3 - CL in log-return space
    crack_cols = {"cl1s", "rb1s", "ho1s"}
    crack_avail = crack_cols.issubset(set(available))

    if crack_avail:
        crack_321 = (2 * lr_wide["rb1s"] + lr_wide["ho1s"]) / 3 - lr_wide["cl1s"]
        crack_z60 = (crack_321 - crack_321.rolling(60).mean()) / crack_321.rolling(60).std()
    else:
        logger.warning("Cannot compute 3-2-1 crack: missing %s", crack_cols - set(available))
        crack_321 = pd.Series(np.nan, index=lr_wide.index)
        crack_z60 = pd.Series(np.nan, index=lr_wide.index)

    crack_321.name = "crack_321_proxy"
    crack_z60.name = "crack_321_z_60d"

    # HO/CL and RB/CL log-price spreads
    ho_cl = (
        np.log(close_wide["ho1s"] / close_wide["cl1s"])
        if {"ho1s", "cl1s"}.issubset(set(available))
        else pd.Series(np.nan, index=close_wide.index)
    )
    rb_cl = (
        np.log(close_wide["rb1s"] / close_wide["cl1s"])
        if {"rb1s", "cl1s"}.issubset(set(available))
        else pd.Series(np.nan, index=close_wide.index)
    )
    ho_cl.name = "ho_cl_spread"
    rb_cl.name = "rb_cl_spread"

    # Cyclical date encoding for seasonality
    date_idx = lr_wide.index
    month = date_idx.month.values.astype(float)
    dow = date_idx.dayofweek.values.astype(float)

    month_sin = pd.Series(np.sin(2 * np.pi * month / 12), index=date_idx, name="month_sin")
    month_cos = pd.Series(np.cos(2 * np.pi * month / 12), index=date_idx, name="month_cos")
    dow_sin = pd.Series(np.sin(2 * np.pi * dow / 5), index=date_idx, name="dow_sin")
    dow_cos = pd.Series(np.cos(2 * np.pi * dow / 5), index=date_idx, name="dow_cos")

    winter = pd.Series(
        ((month >= 11) | (month <= 2)).astype(float), index=date_idx, name="winter_indicator"
    )
    driving = pd.Series(
        ((month >= 5) & (month <= 8)).astype(float), index=date_idx, name="driving_season"
    )

    date_feats = pd.concat(
        [basket, crack_321, crack_z60, ho_cl, rb_cl,
         month_sin, month_cos, dow_sin, dow_cos, winter, driving],
        axis=1,
    )

    # Per-instrument: relative return and rolling correlation to basket
    records = []
    for inst in available:
        inst_lr5 = lr_wide[inst].rolling(5).sum()
        rel_ret = inst_lr5 - basket_ret5
        corr_to_basket = lr_wide[inst].rolling(cfg.corr_window).corr(lr_wide.mean(axis=1))

        per_inst = pd.DataFrame({
            "date": lr_wide.index,
            "instrument": inst,
            "rel_ret_vs_basket_5d": rel_ret.values,
            "corr_to_basket_60d": corr_to_basket.values,
        })
        records.append(per_inst)

    per_inst_df = pd.concat(records, ignore_index=True)

    result = per_inst_df.merge(
        date_feats.reset_index().rename(columns={"index": "date"}),
        on="date", how="left",
    )
    result = result[result["instrument"].isin(energy)].copy()
    result = result.sort_values(["instrument", "date"]).reset_index(drop=True)

    logger.info("Energy features done: %d rows, %d columns", len(result), len(result.columns))
    return result

---
## 4. Tier 2b: Metals-Specific Features

Metals features capture the cross-instrument relationships within the metals complex (GC, SI, HG, PL).

Key features:

- **Gold/Silver ratio (GSR):** Risk-off indicator. When gold outperforms silver (GSR rising), it signals risk aversion.
- **Copper/Gold ratio (CGR):** Growth proxy. Rising copper relative to gold signals risk-on / growth optimism (Gartman, 2010).
- **Gold/Platinum ratio:** Captures the difference between pure safe-haven demand (gold) and partly industrial demand (platinum).
- **Silver/Copper ratio:** Precious vs. industrial split within the non-gold metals.
- **Average pairwise correlation and dispersion:** When all four metals trade together (high mean corr, low dispersion), the move is macro-driven. When they diverge (low mean, high dispersion), moves are more idiosyncratic.

In [13]:
def _wide_close_m(df: pd.DataFrame, instruments: list[str]) -> pd.DataFrame:
    sub = df[df["instrument"].isin(instruments)][["date", "instrument", "close"]].copy()
    return sub.pivot(index="date", columns="instrument", values="close")


def _wide_logret_m(df: pd.DataFrame, instruments: list[str]) -> pd.DataFrame:
    sub = df[df["instrument"].isin(instruments)][["date", "instrument", "logret_1d"]].copy()
    return sub.pivot(index="date", columns="instrument", values="logret_1d")


def compute_metals_features(base_df: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    """
    Compute all Tier 2b metals-specific features.

    Returns a long-format DataFrame with only metals instrument rows.
    """
    metals = cfg.metals_instruments
    precious = cfg.precious_metals
    available = [i for i in metals if i in base_df["instrument"].unique()]

    close_w = _wide_close_m(base_df, available)
    lr_w = _wide_logret_m(base_df, available)

    # Precious metals basket (GC, SI, PL)
    prec_avail = [i for i in precious if i in available]
    basket_ret5 = lr_w[prec_avail].rolling(5).sum().mean(axis=1)
    basket_vol20 = lr_w[prec_avail].rolling(20).std().mean(axis=1)
    basket_ret5.name = "precious_basket_ret_5d"
    basket_vol20.name = "precious_basket_vol_20d"

    # Price ratios
    def _safe_ratio(num: str, den: str) -> pd.Series:
        if num in close_w and den in close_w:
            return close_w[num] / close_w[den].replace(0, np.nan)
        return pd.Series(np.nan, index=close_w.index)

    gsr = _safe_ratio("gc1s", "si1s")
    gsr.name = "gold_silver_ratio"
    gsr_z60 = (gsr - gsr.rolling(60).mean()) / gsr.rolling(60).std()
    gsr_z60.name = "gsr_z_60d"

    gpr = _safe_ratio("gc1s", "pl1s")
    gpr.name = "gold_platinum_ratio"

    scr = _safe_ratio("si1s", "hg1s")
    scr.name = "silver_copper_ratio"

    cgr = _safe_ratio("hg1s", "gc1s")
    cgr.name = "copper_gold_ratio"

    # Pairwise correlation stats across all metals
    dates = lr_w.index
    if len(available) >= 2:
        pairs = [(a, b) for i, a in enumerate(available) for b in available[i + 1:]]
        pair_corrs = pd.DataFrame(index=dates)
        for a, b in pairs:
            pair_corrs[f"{a}_{b}"] = lr_w[a].rolling(cfg.corr_window).corr(lr_w[b])

        avg_pairwise_corr = pair_corrs.mean(axis=1)
        corr_dispersion = pair_corrs.std(axis=1)
    else:
        avg_pairwise_corr = pd.Series(np.nan, index=dates)
        corr_dispersion = pd.Series(np.nan, index=dates)

    avg_pairwise_corr.name = "avg_pairwise_corr_60d"
    corr_dispersion.name = "corr_dispersion_60d"

    date_feats = pd.concat(
        [basket_ret5, basket_vol20, gsr, gsr_z60, gpr, scr, cgr,
         avg_pairwise_corr, corr_dispersion],
        axis=1,
    )

    # Per-instrument: relative return and correlation to precious basket
    basket_lr = lr_w[prec_avail].mean(axis=1)
    records = []
    for inst in available:
        inst_lr5 = lr_w[inst].rolling(5).sum()
        rel_ret = inst_lr5 - basket_ret5
        corr_to_precious = lr_w[inst].rolling(cfg.corr_window).corr(basket_lr)

        per = pd.DataFrame({
            "date": dates,
            "instrument": inst,
            "corr_to_precious_60d": corr_to_precious.values,
            "rel_ret_vs_precious_5d": rel_ret.values,
        })
        records.append(per)

    per_inst_df = pd.concat(records, ignore_index=True)

    result = per_inst_df.merge(
        date_feats.reset_index().rename(columns={"index": "date"}),
        on="date", how="left",
    )
    result = result[result["instrument"].isin(metals)].copy()
    result = result.sort_values(["instrument", "date"]).reset_index(drop=True)

    logger.info("Metals features done: %d rows, %d columns", len(result), len(result.columns))
    return result

---
## 5. Tier 3: Latent Features (GMM, HMM, K-means)

Latent features encode market regime in a way that the metamodel can condition on. All three models are run in a strict walk-forward fashion: the model is fit on past data only and then used to score a future window. This prevents any look-ahead bias.

The three models are:
1. **GMM (Lecture 2):** Gaussian Mixture Model on per-instrument volatility features. Produces soft responsibilities for three regimes: low-vol, mid-vol, and high-vol.
2. **HMM (Lecture 3):** Hidden Markov Model on (logret_1d, vol_20d). Uses the explicit forward algorithm (filtering), not the forward-backward smoother, to avoid leakage. Produces P(calm) and P(turbulent) at each date, plus a one-step-ahead forecast.
3. **K-means (Lecture 2):** Hard cross-sectional clustering on (ret_5d, vol_20d, volume_z_20d). Produces a regime cluster ID and cluster size (fraction of instruments in the same cluster) at each date.

### 5.1 Walk-Forward GMM

In [14]:
from typing import NamedTuple
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM


class _GMMModel(NamedTuple):
    gmm: GaussianMixture
    perm: np.ndarray  # index permutation: maps sorted component index to original


def _fit_gmm(
    X: np.ndarray,
    n_components: int,
    sort_col_idx: int,
    random_state: int,
) -> _GMMModel:
    """
    Fit a GMM and compute a label-stabilising permutation.

    After fitting, components are sorted by the mean of sort_col_idx in
    ascending order. This ensures component 0 always maps to the lowest-vol
    regime across different refits, even though GMM labels are arbitrary.
    """
    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type="full",
        random_state=random_state,
        max_iter=300,
        n_init=5,
    )
    gmm.fit(X)
    component_means = gmm.means_[:, sort_col_idx]
    perm = np.argsort(component_means)
    return _GMMModel(gmm=gmm, perm=perm)


def _score_gmm(model: _GMMModel, X: np.ndarray) -> np.ndarray:
    """Return responsibilities reordered by the stable permutation (low to high vol)."""
    resp = model.gmm.predict_proba(X)
    return resp[:, model.perm]


def walk_forward_gmm_responsibilities(
    features: pd.DataFrame,
    n_components: int = 3,
    refit_freq: int = 63,
    warmup: int = 504,
    sort_col_idx: int = 0,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Walk-forward GMM posterior responsibilities.

    At each refit date, GaussianMixture is fit on features[:fit_date] (past
    data only). Responsibilities for [fit_date:next_refit_date] are scored
    on the frozen model.

    Returns a DataFrame with columns resp_0, resp_1, resp_2 (sorted
    low vol to high vol). Rows before the first refit are NaN.
    """
    np.random.seed(random_state)

    n = len(features)
    cols = [f"resp_{i}" for i in range(n_components)]
    out = pd.DataFrame(np.nan, index=features.index, columns=cols)

    dates = features.index
    X_all = features.values.astype(float)

    first_refit = warmup
    if first_refit >= n:
        logger.warning("GMM: not enough data for warmup (%d rows, warmup=%d)", n, warmup)
        return out

    refit_dates_idx = list(range(first_refit, n, refit_freq))
    current_model: _GMMModel | None = None
    next_refit_ptr = 0

    for t in range(n):
        if next_refit_ptr < len(refit_dates_idx) and t == refit_dates_idx[next_refit_ptr]:
            fit_end = t
            X_train = X_all[:fit_end]
            valid_mask = ~np.any(np.isnan(X_train), axis=1)
            X_clean = X_train[valid_mask]

            if len(X_clean) >= n_components * 10:
                current_model = _fit_gmm(X_clean, n_components, sort_col_idx, random_state)
                logger.info(
                    "GMM refit at index %d (date %s), N_train=%d",
                    t,
                    dates[t].date() if hasattr(dates[t], "date") else dates[t],
                    len(X_clean),
                )
            next_refit_ptr += 1

        if current_model is None:
            continue

        x_t = X_all[t : t + 1]
        if np.any(np.isnan(x_t)):
            continue

        resp = _score_gmm(current_model, x_t)
        out.iloc[t] = resp[0]

    return out

### 5.2 Walk-Forward HMM with Explicit Forward Algorithm

The forward algorithm computes the filtering posterior: $p(H_t \mid x_{1:t})$. This uses only observations up to and including time $t$, so there is no future leakage.

The recursion is:
$$\alpha_1 = \pi \odot \Gamma(x_1)$$
$$\alpha_t = (Q^\top \alpha_{t-1}) \odot \Gamma(x_t), \quad t = 2, \ldots, T$$
$$\hat{\alpha}_t = \alpha_t / \sum \alpha_t$$

where $\pi$ is the initial state distribution, $Q$ is the transition matrix, and $\Gamma(x_t)$ is the emission likelihood vector.

This is deliberately *not* the forward-backward (smoothing) algorithm. `hmmlearn.predict_proba()` runs forward-backward and would introduce leakage by using future observations.

In [15]:
class _HMMModel(NamedTuple):
    hmm: GaussianHMM
    perm: np.ndarray        # stable label permutation (0=calm, 1=turbulent)
    scaler: StandardScaler  # fit on training data; applied at score time


def _fit_hmm(X: np.ndarray, n_states: int, random_state: int) -> _HMMModel:
    """
    Fit a GaussianHMM and compute a label-stabilising permutation.

    The turbulent state is defined as the one with the highest mean
    absolute log return. This assignment is stable across refits.

    Inputs are standardised before fitting to prevent large-scale features
    (e.g. natural gas vs. crude) from dominating the emission likelihood.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = GaussianHMM(
        n_components=n_states,
        covariance_type="diag",
        n_iter=200,
        random_state=random_state,
        tol=1e-4,
        min_covar=1e-3,
    )
    model.fit(X_scaled)

    abs_mean_ret = np.abs(model.means_[:, 0])
    perm = np.argsort(abs_mean_ret)  # perm[0]=calm, perm[1]=turbulent
    return _HMMModel(hmm=model, perm=perm, scaler=scaler)


def _log_sum_exp(a: np.ndarray) -> float:
    """Numerically stable log-sum-exp over a 1D array."""
    m = a.max()
    return m + np.log(np.sum(np.exp(a - m)))


def _log_sum_exp_vec(a: np.ndarray, axis: int) -> np.ndarray:
    """Numerically stable log-sum-exp along a given axis."""
    m = a.max(axis=axis, keepdims=True)
    return m.squeeze(axis=axis) + np.log(np.sum(np.exp(a - m), axis=axis))


def _forward_algorithm(model: GaussianHMM, X: np.ndarray) -> np.ndarray:
    """
    Explicit forward algorithm: filtering posteriors p(H_t | x_{1:t}).

    Returns shape (T, n_states). This is FILTERING only. Do not use
    hmmlearn.predict_proba(), which runs forward-backward (smoothing)
    and introduces leakage from future observations.
    """
    T = len(X)
    n_states = model.n_components
    log_transmat = np.log(model.transmat_ + 1e-300)
    log_emission = model._compute_log_likelihood(X)  # shape (T, n_states)

    alpha = np.zeros((T, n_states))

    log_alpha = np.log(model.startprob_ + 1e-300) + log_emission[0]
    log_alpha -= _log_sum_exp(log_alpha)
    alpha[0] = np.exp(log_alpha)

    for t in range(1, T):
        log_alpha = (
            _log_sum_exp_vec(log_transmat.T + log_alpha[:, None], axis=0)
            + log_emission[t]
        )
        log_alpha -= _log_sum_exp(log_alpha)
        alpha[t] = np.exp(log_alpha)

    return alpha


def _compute_regime_age(history: list[int]) -> int:
    """Number of consecutive bars the current hard regime has persisted."""
    if not history:
        return 0
    current = history[-1]
    age = 0
    for val in reversed(history[:-1]):
        if val == current:
            age += 1
        else:
            break
    return age


def walk_forward_hmm_filtering(
    features: pd.DataFrame,
    n_states: int = 2,
    refit_freq: int = 63,
    warmup: int = 504,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Walk-forward HMM filtering posteriors and derived features.

    At each refit:
      1. Fit GaussianHMM on features[:fit_date] (strictly past data).
      2. Run the explicit forward algorithm on [fit_date:next_refit_date].
      3. Compute one-step-ahead forecast: P(turbulent at t+1) = alpha_t @ Q.
      4. Track hard regime (argmax) for regime_age.

    Features produced:
        hmm_filt_calm       : P(calm | x_{1:t})
        hmm_filt_turbulent  : P(turbulent | x_{1:t})
        hmm_next_turbulent  : P(turbulent at t+1) via one-step forecast
        hmm_regime_age      : bars since last hard regime switch
    """
    np.random.seed(random_state)

    n = len(features)
    out = pd.DataFrame(
        np.nan,
        index=features.index,
        columns=["hmm_filt_calm", "hmm_filt_turbulent", "hmm_next_turbulent", "hmm_regime_age"],
    )
    X_all = features.values.astype(float)

    refit_idx = list(range(warmup, n, refit_freq))
    if not refit_idx:
        logger.warning("HMM: insufficient data for warmup.")
        return out

    next_refit_ptr = 0
    current_model: _HMMModel | None = None
    hard_regime_history: list[int] = []

    t = 0
    while t < n:
        if next_refit_ptr < len(refit_idx) and t == refit_idx[next_refit_ptr]:
            X_train = X_all[:t]
            valid = ~np.any(np.isnan(X_train), axis=1)
            X_clean = X_train[valid]

            if len(X_clean) >= n_states * 20:
                current_model = _fit_hmm(X_clean, n_states, random_state)
                logger.info(
                    "HMM refit at index %d (date %s), N_train=%d",
                    t,
                    features.index[t].date() if hasattr(features.index[t], "date") else t,
                    len(X_clean),
                )
            next_refit_ptr += 1

        if current_model is None:
            hard_regime_history.append(-1)
            t += 1
            continue

        next_refit = refit_idx[next_refit_ptr] if next_refit_ptr < len(refit_idx) else n
        end = min(next_refit, n)

        X_window = X_all[t:end]
        valid_rows = ~np.any(np.isnan(X_window), axis=1)

        if not np.any(valid_rows):
            hard_regime_history.extend([-1] * (end - t))
            t = end
            continue

        X_window_scaled = current_model.scaler.transform(X_window)

        try:
            alpha = _forward_algorithm(current_model.hmm, X_window_scaled)
        except Exception as exc:
            logger.warning("Forward algorithm failed for window [%d:%d]: %s", t, end, exc)
            hard_regime_history.extend([-1] * (end - t))
            t = end
            continue

        alpha_sorted = alpha[:, current_model.perm]

        for i, abs_t in enumerate(range(t, end)):
            if not valid_rows[i]:
                hard_regime_history.append(-1)
                continue

            alpha_t = alpha_sorted[i]
            calm = float(alpha_t[0])
            turb = float(alpha_t[1])

            Q_sorted = current_model.hmm.transmat_[current_model.perm, :][:, current_model.perm]
            next_proba = alpha_t @ Q_sorted
            next_turb = float(next_proba[1])

            hard = int(np.argmax(alpha_t))
            hard_regime_history.append(hard)

            regime_age = _compute_regime_age(hard_regime_history)
            out.iloc[abs_t] = [calm, turb, next_turb, float(regime_age)]

        t = end

    return out

### 5.3 Basket HMM

In addition to per-instrument HMMs, we run a basket HMM on the cross-sectional mean of log returns and vol for each asset class. This separates asset-class-wide regime shifts (e.g. a broad energy selloff) from idiosyncratic per-instrument moves.

In [16]:
def walk_forward_basket_hmm(
    base_df: pd.DataFrame,
    instruments: list[str],
    label: str,
    n_states: int = 2,
    refit_freq: int = 63,
    warmup: int = 504,
    random_state: int = 42,
) -> pd.Series:
    """
    Walk-forward HMM on the cross-sectional basket mean (logret, vol).

    By modelling the basket rather than individual instruments, we capture
    asset-class-wide regime shifts separately from idiosyncratic moves.

    Returns a date-indexed Series of P(turbulent) for the basket.
    """
    avail = [i for i in instruments if i in base_df["instrument"].unique()]

    lr_wide = base_df[base_df["instrument"].isin(avail)].pivot_table(
        index="date", columns="instrument", values="logret_1d"
    )
    vol_wide = base_df[base_df["instrument"].isin(avail)].pivot_table(
        index="date", columns="instrument", values="vol_20d"
    )

    basket_lr = lr_wide.mean(axis=1)
    basket_vol = vol_wide.mean(axis=1)

    feats = pd.DataFrame({"logret_1d": basket_lr, "vol_20d": basket_vol}).dropna()

    hmm_out = walk_forward_hmm_filtering(
        feats,
        n_states=n_states,
        refit_freq=refit_freq,
        warmup=warmup,
        random_state=random_state,
    )

    turb = hmm_out["hmm_filt_turbulent"]
    turb.name = label
    return turb

### 5.4 Walk-Forward K-means

K-means gives hard cluster assignments on the cross-section of instruments. At each date, all instruments are mapped to one of K=3 clusters based on (ret_5d, vol_20d, volume_z_20d). The cluster_size feature captures how much the asset class is moving together versus dispersing.

In [17]:
def walk_forward_kmeans(
    base_df: pd.DataFrame,
    instruments: list[str],
    refit_freq: int = 63,
    warmup: int = 504,
    k: int = 3,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Walk-forward cross-sectional K-means clustering.

    At each date, cluster all instruments on [ret_5d, vol_20d, volume_z_20d].
    Centroids are refit quarterly on past data only.

    Features produced:
        kmeans_cluster_id   : hard cluster assignment (0, 1, or 2)
        kmeans_cluster_size : fraction of instruments assigned to the same cluster
    """
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler as SS

    np.random.seed(random_state)

    feature_cols = ["ret_5d", "vol_20d", "volume_z_20d"]
    avail = [i for i in instruments if i in base_df["instrument"].unique()]

    wide = {}
    for col in feature_cols:
        wide[col] = base_df[base_df["instrument"].isin(avail)].pivot_table(
            index="date", columns="instrument", values=col
        )

    dates = wide[feature_cols[0]].index
    n_dates = len(dates)

    records = []
    km_model: KMeans | None = None
    scaler: SS | None = None
    refit_dates_idx = list(range(warmup, n_dates, refit_freq))
    next_refit_ptr = 0

    for t_idx, date in enumerate(dates):
        if next_refit_ptr < len(refit_dates_idx) and t_idx == refit_dates_idx[next_refit_ptr]:
            panels = []
            for col in feature_cols:
                vals = wide[col].iloc[:t_idx].values.flatten()
                panels.append(vals)
            X_train = np.column_stack(panels)
            valid = ~np.any(np.isnan(X_train), axis=1)
            X_clean = X_train[valid]

            if len(X_clean) >= k * 10:
                scaler = SS().fit(X_clean)
                km_model = KMeans(n_clusters=k, random_state=random_state, n_init=10)
                km_model.fit(scaler.transform(X_clean))
                logger.info("K-means refit at index %d (date %s)", t_idx, date)

            next_refit_ptr += 1

        if km_model is None or scaler is None:
            for inst in avail:
                records.append({
                    "date": date, "instrument": inst,
                    "kmeans_cluster_id": np.nan,
                    "kmeans_cluster_size": np.nan,
                })
            continue

        row_feats = []
        for inst in avail:
            feat_row = [
                wide[col].loc[date, inst] if inst in wide[col].columns else np.nan
                for col in feature_cols
            ]
            row_feats.append(feat_row)

        X_cross = np.array(row_feats, dtype=float)
        valid = ~np.any(np.isnan(X_cross), axis=1)
        labels = np.full(len(avail), np.nan)

        if np.any(valid):
            labels[valid] = km_model.predict(scaler.transform(X_cross[valid])).astype(float)

        for i, inst in enumerate(avail):
            cid = labels[i]
            csize = float(np.sum(labels == cid) / len(avail)) if not np.isnan(cid) else np.nan
            records.append({
                "date": date, "instrument": inst,
                "kmeans_cluster_id": cid,
                "kmeans_cluster_size": csize,
            })

    return pd.DataFrame(records)

### 5.5 Latent Feature Orchestrator

In [18]:
def compute_latent_features(base_df: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    """
    Compute all Tier 3 latent features: GMM, HMM, and K-means.

    Per-instrument:
        gmm_resp_low_vol, gmm_resp_med_vol, gmm_resp_high_vol, gmm_max_resp
        hmm_filt_calm, hmm_filt_turbulent, hmm_next_turbulent, hmm_regime_age

    Date-level (same value for all instruments in the asset class):
        hmm_basket_turbulent_energy
        hmm_basket_turbulent_metals

    Per (date, instrument):
        kmeans_cluster_id, kmeans_cluster_size
    """
    logger.info("Computing latent features...")
    all_instruments = cfg.energy_instruments + cfg.metals_instruments
    available = [i for i in all_instruments if i in base_df["instrument"].unique()]

    records_list = []

    for inst in available:
        g = base_df[base_df["instrument"] == inst].sort_values("date").copy().reset_index(drop=True)

        # GMM on [vol_20d, vol_of_vol_20d, parkinson_20d]
        gmm_feat_cols = ["vol_20d", "vol_of_vol_20d", "parkinson_20d"]
        gmm_feats = g[gmm_feat_cols].copy()
        gmm_feats.index = g["date"]

        gmm_resp = walk_forward_gmm_responsibilities(
            gmm_feats,
            n_components=cfg.gmm_n_components,
            refit_freq=cfg.refit_frequency,
            warmup=cfg.warmup,
            sort_col_idx=0,
            random_state=cfg.random_state,
        )
        gmm_resp.columns = ["gmm_resp_low_vol", "gmm_resp_med_vol", "gmm_resp_high_vol"]
        gmm_resp["gmm_max_resp"] = gmm_resp.idxmax(axis=1).map(
            {"gmm_resp_low_vol": 0.0, "gmm_resp_med_vol": 1.0, "gmm_resp_high_vol": 2.0}
        )

        # HMM on [logret_1d, vol_20d]
        hmm_feat_cols = ["logret_1d", "vol_20d"]
        hmm_feats = g[hmm_feat_cols].copy()
        hmm_feats.index = g["date"]

        hmm_out = walk_forward_hmm_filtering(
            hmm_feats,
            n_states=cfg.hmm_n_states,
            refit_freq=cfg.refit_frequency,
            warmup=cfg.warmup,
            random_state=cfg.random_state,
        )

        inst_df = g[["date"]].copy()
        inst_df["instrument"] = inst
        inst_df = inst_df.join(gmm_resp.reset_index(drop=True))
        inst_df = inst_df.join(hmm_out.reset_index(drop=True))

        records_list.append(inst_df)

    result = pd.concat(records_list, ignore_index=True)

    # Basket HMMs: one turbulence probability per date per asset class
    energy_basket_turb = walk_forward_basket_hmm(
        base_df, cfg.energy_instruments,
        label="hmm_basket_turbulent_energy",
        n_states=cfg.hmm_n_states,
        refit_freq=cfg.refit_frequency,
        warmup=cfg.warmup,
        random_state=cfg.random_state,
    )
    metals_basket_turb = walk_forward_basket_hmm(
        base_df, cfg.metals_instruments,
        label="hmm_basket_turbulent_metals",
        n_states=cfg.hmm_n_states,
        refit_freq=cfg.refit_frequency,
        warmup=cfg.warmup,
        random_state=cfg.random_state,
    )

    basket_df = pd.DataFrame({
        "date": energy_basket_turb.index,
        "hmm_basket_turbulent_energy": energy_basket_turb.values,
        "hmm_basket_turbulent_metals": metals_basket_turb.reindex(energy_basket_turb.index).values,
    })

    result = result.merge(basket_df, on="date", how="left")

    # K-means per asset class
    energy_km = walk_forward_kmeans(
        base_df, cfg.energy_instruments,
        refit_freq=cfg.refit_frequency,
        warmup=cfg.warmup,
        k=cfg.kmeans_k,
        random_state=cfg.random_state,
    )
    metals_km = walk_forward_kmeans(
        base_df, cfg.metals_instruments,
        refit_freq=cfg.refit_frequency,
        warmup=cfg.warmup,
        k=cfg.kmeans_k,
        random_state=cfg.random_state,
    )
    km_all = pd.concat([energy_km, metals_km], ignore_index=True)

    result = result.merge(km_all, on=["date", "instrument"], how="left")
    result = result.sort_values(["instrument", "date"]).reset_index(drop=True)

    logger.info("Latent features done: %d rows, %d columns", len(result), len(result.columns))
    return result

---
## 6. Tier 4: Primary Signal Features

These features encode the primary model signal's own history for meta-labeling. The metamodel needs to learn *when* the primary signal is reliable, so these features describe how confident and consistent the primary model has been recently.

- **Lags (1, 5):** Yesterday's and last week's signal. Captures autocorrelation in the primary model's decisions.
- **Streak:** Consecutive bars without a sign flip. A long streak means the primary model has been committed to a direction.
- **Agreement (5d):** Rolling mean of the signal over 5 days. High absolute value means the model has been consistently directional.

All operations are backward-looking only.

In [19]:
def compute_primary_signal_features(signals: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    """
    Compute Tier 4 primary-signal-aware features.

    Input: long-format signals with columns [date, instrument, primary_signal].
    primary_signal is in {-1, 0, +1}.

    Features produced:
        primary_signal_lag1          : signal shifted by 1 day
        primary_signal_lag5          : signal shifted by 5 days
        primary_signal_streak        : bars since last sign flip
        primary_signal_agreement_5d  : 5-day rolling mean of signal
    """
    out_parts = []

    for inst, g in signals.groupby("instrument", sort=False):
        g = g.sort_values("date").reset_index(drop=True).copy()
        sig = g["primary_signal"].astype(float)

        for lag in cfg.signal_lags:
            g[f"primary_signal_lag{lag}"] = sig.shift(lag)

        # Streak: bars since last sign flip
        # Zero signal is treated as continuing the previous non-zero sign
        nonzero_sign = np.sign(sig.replace(0, np.nan).ffill().fillna(0))
        flips = (nonzero_sign != nonzero_sign.shift(1)).astype(int)
        flip_groups = flips.cumsum()
        g["primary_signal_streak"] = flip_groups.groupby(flip_groups).cumcount()

        g["primary_signal_agreement_5d"] = sig.rolling(cfg.signal_agreement_window).mean()

        out_parts.append(g)

    result = pd.concat(out_parts, ignore_index=True)
    result = result.sort_values(["instrument", "date"]).reset_index(drop=True)
    logger.info("Primary signal features done: %d rows", len(result))
    return result

---
## 7. Feature Dictionary

A machine-readable record of every feature column: its formula, course bucket, and purpose.

In [20]:
_FEATURE_DICT_ROWS: list[dict] = [
    # Tier 1: returns
    {"name": "ret_1d",            "formula_or_source": "close.pct_change()",                         "course_bucket": "base",        "intent": "Daily pct return; primary momentum signal"},
    {"name": "logret_1d",         "formula_or_source": "log(close/close.shift(1))",                  "course_bucket": "base",        "intent": "Log return; additive over horizons, approx normal"},
    {"name": "ret_5d",            "formula_or_source": "logret_1d.rolling(5).sum()",                 "course_bucket": "base",        "intent": "5-day cumulative log return; weekly momentum"},
    {"name": "ret_10d",           "formula_or_source": "logret_1d.rolling(10).sum()",                "course_bucket": "base",        "intent": "10-day cumulative log return; bi-weekly momentum"},
    {"name": "ret_20d",           "formula_or_source": "logret_1d.rolling(20).sum()",                "course_bucket": "base",        "intent": "20-day cumulative log return; monthly momentum"},
    {"name": "ret_60d",           "formula_or_source": "logret_1d.rolling(60).sum()",                "course_bucket": "base",        "intent": "60-day cumulative log return; quarterly momentum"},
    # Tier 1: volatility
    {"name": "vol_20d",           "formula_or_source": "logret_1d.rolling(20).std()",                "course_bucket": "base",        "intent": "20-day realised vol; regime conditioning"},
    {"name": "vol_60d",           "formula_or_source": "logret_1d.rolling(60).std()",                "course_bucket": "base",        "intent": "60-day realised vol; long-run vol baseline"},
    {"name": "parkinson_20d",     "formula_or_source": "sqrt(sum(log(H/L)^2)/(4*ln2*N))",           "course_bucket": "base",        "intent": "Parkinson HL vol; more efficient than CC vol"},
    {"name": "garman_klass_20d",  "formula_or_source": "sqrt(0.5*(log H/L)^2-(2ln2-1)*(log C/O)^2)","course_bucket": "base",        "intent": "Garman-Klass OHLC vol estimator"},
    {"name": "vol_of_vol_20d",    "formula_or_source": "vol_20d.rolling(20).std()",                  "course_bucket": "base",        "intent": "Volatility-of-vol; signals regime transitions"},
    {"name": "vol_ratio_5_60",    "formula_or_source": "vol_5d / vol_60d",                           "course_bucket": "base",        "intent": "Short/long vol ratio; >1 signals vol expansion"},
    # Tier 1: volume/OI
    {"name": "volume_z_20d",      "formula_or_source": "(volume - roll_mean) / roll_std, W=20",      "course_bucket": "base",        "intent": "Volume anomaly; spikes precede breakouts"},
    {"name": "oi_z_60d",          "formula_or_source": "(OI - roll_mean) / roll_std, W=60",          "course_bucket": "base",        "intent": "OI z-score; positioning build/unwind"},
    {"name": "oi_change_5d",      "formula_or_source": "OI.pct_change(5)",                           "course_bucket": "base",        "intent": "5-day OI change; contract roll or new positioning"},
    {"name": "volume_oi_ratio",   "formula_or_source": "volume / open_interest",                     "course_bucket": "base",        "intent": "Turnover ratio; high = speculative activity"},
    # Tier 1: technical indicators
    {"name": "rsi_14",            "formula_or_source": "Wilder RSI, period=14",                      "course_bucket": "TA",          "intent": "Overbought/oversold momentum oscillator"},
    {"name": "macd",              "formula_or_source": "EMA(12) - EMA(26)",                          "course_bucket": "TA",          "intent": "MACD line; trend-following signal"},
    {"name": "macd_signal",       "formula_or_source": "EMA(macd, 9)",                               "course_bucket": "TA",          "intent": "MACD signal line; crossover trigger"},
    {"name": "macd_hist",         "formula_or_source": "macd - macd_signal",                         "course_bucket": "TA",          "intent": "MACD histogram; momentum acceleration"},
    {"name": "bb_width_20",       "formula_or_source": "(upper-lower)/mid, W=20",                    "course_bucket": "TA",          "intent": "Bollinger Band width; vol proxy for squeeze breakouts"},
    {"name": "bb_pos_20",         "formula_or_source": "(close-lower)/(upper-lower), W=20",          "course_bucket": "TA",          "intent": "Bollinger %B; relative position within band"},
    {"name": "atr_14",            "formula_or_source": "Wilder ATR, period=14",                      "course_bucket": "TA",          "intent": "True range; used by Part 2 for barrier sizing"},
    {"name": "adx_14",            "formula_or_source": "Wilder ADX, period=14",                      "course_bucket": "TA",          "intent": "Trend strength; ADX>25 signals trending regime"},
    # Tier 1: drawdown
    {"name": "dd_from_60d_high",       "formula_or_source": "close/rolling_max(60)-1",           "course_bucket": "base",        "intent": "Drawdown from 60d high; tail-risk conditioning"},
    {"name": "runup_from_60d_low",     "formula_or_source": "close/rolling_min(60)-1",           "course_bucket": "base",        "intent": "Run-up from 60d low; momentum from trough"},
    {"name": "days_since_60d_high",    "formula_or_source": "bars since rolling-60 argmax",      "course_bucket": "base",        "intent": "Trend staleness; long lag = weakening trend"},
    # Tier 1: trend scanning
    {"name": "trend_tstat_10",    "formula_or_source": "OLS slope t-stat, H=10, backward",          "course_bucket": "Lecture 1",   "intent": "Short-term trend strength"},
    {"name": "trend_tstat_20",    "formula_or_source": "OLS slope t-stat, H=20, backward",          "course_bucket": "Lecture 1",   "intent": "Medium-term trend strength"},
    {"name": "trend_tstat_60",    "formula_or_source": "OLS slope t-stat, H=60, backward",          "course_bucket": "Lecture 1",   "intent": "Long-term trend strength"},
    {"name": "trend_tstat_best",  "formula_or_source": "argmax |t| across H in {10,20,60}",         "course_bucket": "Lecture 1",   "intent": "Best-fit trend t-stat (max-|t| selection)"},
    {"name": "trend_H_best",      "formula_or_source": "H at argmax |t|",                           "course_bucket": "Lecture 1",   "intent": "Optimal trend horizon; characterises trend duration"},
    # Tier 2a: energy
    {"name": "energy_basket_ret_5d",   "formula_or_source": "cross-sectional mean of 5d logret",    "course_bucket": "domain (energy)", "intent": "Energy complex 5d momentum"},
    {"name": "energy_basket_vol_20d",  "formula_or_source": "cross-sectional mean of 20d vol",      "course_bucket": "domain (energy)", "intent": "Energy complex volatility level"},
    {"name": "rel_ret_vs_basket_5d",   "formula_or_source": "inst_5d_ret - basket_5d_ret",          "course_bucket": "domain (energy)", "intent": "Relative momentum vs energy basket"},
    {"name": "corr_to_basket_60d",     "formula_or_source": "60d rolling corr with basket mean ret","course_bucket": "domain (energy)", "intent": "Idiosyncratic vs systematic energy exposure"},
    {"name": "crack_321_proxy",        "formula_or_source": "(2*rb_ret+ho_ret)/3 - cl_ret",         "course_bucket": "domain (energy)", "intent": "3-2-1 crack spread return; refinery margin proxy"},
    {"name": "crack_321_z_60d",        "formula_or_source": "z-score of crack_321, W=60",           "course_bucket": "domain (energy)", "intent": "Crack spread anomaly; drives energy complex moves"},
    {"name": "ho_cl_spread",           "formula_or_source": "log(ho_close/cl_close)",               "course_bucket": "domain (energy)", "intent": "Heating oil premium; seasonal demand indicator"},
    {"name": "rb_cl_spread",           "formula_or_source": "log(rb_close/cl_close)",               "course_bucket": "domain (energy)", "intent": "Gasoline premium; driving season indicator"},
    {"name": "month_sin",              "formula_or_source": "sin(2*pi*month/12)",                   "course_bucket": "seasonality",    "intent": "Cyclical month encoding"},
    {"name": "month_cos",              "formula_or_source": "cos(2*pi*month/12)",                   "course_bucket": "seasonality",    "intent": "Cyclical month encoding"},
    {"name": "dow_sin",                "formula_or_source": "sin(2*pi*dow/5)",                      "course_bucket": "seasonality",    "intent": "Day-of-week cycle"},
    {"name": "dow_cos",                "formula_or_source": "cos(2*pi*dow/5)",                      "course_bucket": "seasonality",    "intent": "Day-of-week cycle"},
    {"name": "winter_indicator",       "formula_or_source": "1 if month in {11,12,1,2} else 0",    "course_bucket": "seasonality",    "intent": "Heating demand season flag"},
    {"name": "driving_season",         "formula_or_source": "1 if month in {5,6,7,8} else 0",      "course_bucket": "seasonality",    "intent": "Gasoline demand season flag"},
    # Tier 2b: metals
    {"name": "precious_basket_ret_5d", "formula_or_source": "mean 5d logret of gc/si/pl",           "course_bucket": "domain (metals)", "intent": "Precious metals momentum"},
    {"name": "precious_basket_vol_20d","formula_or_source": "mean 20d vol of gc/si/pl",             "course_bucket": "domain (metals)", "intent": "Precious metals vol level"},
    {"name": "corr_to_precious_60d",   "formula_or_source": "60d corr with mean precious ret",      "course_bucket": "domain (metals)", "intent": "Precious vs idiosyncratic exposure"},
    {"name": "rel_ret_vs_precious_5d", "formula_or_source": "inst_5d_ret - precious_basket_5d_ret", "course_bucket": "domain (metals)", "intent": "Relative momentum vs precious basket"},
    {"name": "gold_silver_ratio",      "formula_or_source": "gc_close / si_close",                  "course_bucket": "domain (metals)", "intent": "Risk-off indicator; high GSR = gold outperforms"},
    {"name": "gsr_z_60d",              "formula_or_source": "z-score of GSR, W=60",                 "course_bucket": "domain (metals)", "intent": "GSR anomaly; mean-reversion signal"},
    {"name": "gold_platinum_ratio",    "formula_or_source": "gc_close / pl_close",                  "course_bucket": "domain (metals)", "intent": "Macro sentiment; pl has industrial component"},
    {"name": "silver_copper_ratio",    "formula_or_source": "si_close / hg_close",                  "course_bucket": "domain (metals)", "intent": "Precious/industrial split"},
    {"name": "copper_gold_ratio",      "formula_or_source": "hg_close / gc_close",                  "course_bucket": "domain (metals)", "intent": "Global growth proxy (Gartman); risk-on/off signal"},
    {"name": "avg_pairwise_corr_60d",  "formula_or_source": "mean off-diagonal of 4-metal 60d corr matrix","course_bucket": "domain (metals)", "intent": "Systemic co-movement; high = macro-driven regime"},
    {"name": "corr_dispersion_60d",    "formula_or_source": "std of off-diagonal 60d corr entries",  "course_bucket": "domain (metals)", "intent": "Regime heterogeneity across metals"},
    # Tier 3: GMM
    {"name": "gmm_resp_low_vol",       "formula_or_source": "GMM 3-comp posterior, comp 0 (low vol)","course_bucket": "Lecture 2 (GMM)", "intent": "P(low-vol regime); soft cluster membership"},
    {"name": "gmm_resp_med_vol",       "formula_or_source": "GMM 3-comp posterior, comp 1 (mid vol)","course_bucket": "Lecture 2 (GMM)", "intent": "P(medium-vol regime)"},
    {"name": "gmm_resp_high_vol",      "formula_or_source": "GMM 3-comp posterior, comp 2 (high vol)","course_bucket": "Lecture 2 (GMM)", "intent": "P(high-vol regime); risk-off warning"},
    {"name": "gmm_max_resp",           "formula_or_source": "argmax of GMM responsibilities",       "course_bucket": "Lecture 2 (GMM)", "intent": "Hard vol-regime label (0/1/2)"},
    # Tier 3: HMM
    {"name": "hmm_filt_calm",          "formula_or_source": "HMM filtering p(calm | x_{1:t})",      "course_bucket": "Lecture 3 (HMM)", "intent": "Calm-regime filtering posterior; no leakage"},
    {"name": "hmm_filt_turbulent",     "formula_or_source": "HMM filtering p(turbulent | x_{1:t})", "course_bucket": "Lecture 3 (HMM)", "intent": "Turbulent-regime posterior; key risk feature"},
    {"name": "hmm_next_turbulent",     "formula_or_source": "alpha_t @ Q (one-step forecast)",      "course_bucket": "Lecture 3 (HMM)", "intent": "P(turbulent at t+1); predictive risk signal"},
    {"name": "hmm_regime_age",         "formula_or_source": "bars since hard-regime last switched",  "course_bucket": "Lecture 3 (HMM)", "intent": "Regime persistence; long age = stable regime"},
    {"name": "hmm_basket_turbulent_energy","formula_or_source": "basket HMM on energy mean logret/vol","course_bucket": "Lecture 3 (basket HMM)", "intent": "Energy-wide turbulence"},
    {"name": "hmm_basket_turbulent_metals","formula_or_source": "basket HMM on metals mean logret/vol","course_bucket": "Lecture 3 (basket HMM)", "intent": "Metals-wide turbulence"},
    # Tier 3: K-means
    {"name": "kmeans_cluster_id",      "formula_or_source": "K=3 K-means on [ret_5d, vol_20d, volume_z_20d]","course_bucket": "Lecture 2 (K-means)", "intent": "Hard cross-sectional cluster; categorical regime label"},
    {"name": "kmeans_cluster_size",    "formula_or_source": "fraction of instruments in same cluster","course_bucket": "Lecture 2 (K-means)", "intent": "Cluster homogeneity; high = co-movement day"},
    # Tier 4: primary signal
    {"name": "primary_signal",              "formula_or_source": "primary_signals.csv direct",      "course_bucket": "meta-labeling", "intent": "Primary model signal {-1,0,+1}"},
    {"name": "primary_signal_lag1",         "formula_or_source": "primary_signal.shift(1)",         "course_bucket": "meta-labeling", "intent": "Signal yesterday"},
    {"name": "primary_signal_lag5",         "formula_or_source": "primary_signal.shift(5)",         "course_bucket": "meta-labeling", "intent": "Signal 5 days ago"},
    {"name": "primary_signal_streak",       "formula_or_source": "bars since last sign flip",       "course_bucket": "meta-labeling", "intent": "Signal conviction length"},
    {"name": "primary_signal_agreement_5d", "formula_or_source": "rolling mean of signal, W=5",    "course_bucket": "meta-labeling", "intent": "5-day signal agreement in [-1, 1]"},
]


def _build_feature_dictionary() -> pd.DataFrame:
    return pd.DataFrame(_FEATURE_DICT_ROWS)

---
## 8. Pipeline Orchestrator

This cell runs the full pipeline end to end. Steps:

1. Load OHLCV and primary signals from disk
2. Compute base features (all instruments)
3. Compute energy-specific features
4. Compute metals-specific features
5. Compute latent features (GMM, HMM, K-means)
6. Compute primary signal features
7. Merge everything on (date, instrument)
8. Filter to the signal date range and drop raw OHLCV columns
9. Run a partial-NaN sanity check per (instrument, feature) post-warmup
10. Save to Parquet and write the feature dictionary to CSV

In [28]:
def run_pipeline(cfg: FeatureConfig | None = None) -> pd.DataFrame:
    """
    Run the full feature engineering pipeline.

    Returns the final long-format feature matrix, also saved to
    features/features.parquet.
    """
    if cfg is None:
        cfg = FeatureConfig()

    logger.info("=== Feature Engineering Pipeline ===")

    # Step 1: Load data
    ohlcv = load_ohlcv(cfg)
    signals = load_signals(cfg)

    # Step 2: Base features (all instruments)
    base = compute_base_features(ohlcv, cfg)

    # Step 3-4: Asset-class-specific features
    energy_feats = compute_energy_features(base, cfg)
    metals_feats = compute_metals_features(base, cfg)

    # Step 5: Latent features
    latent = compute_latent_features(base, cfg)

    # Step 6: Primary signal features
    sig_feats = compute_primary_signal_features(signals, cfg)

    # Step 7: Merge all on (date, instrument)
    logger.info("Assembling final feature matrix...")
    result = base.copy()
    result = result.merge(energy_feats, on=["date", "instrument"], how="left")
    result = result.merge(metals_feats, on=["date", "instrument"], how="left")
    result = result.merge(latent, on=["date", "instrument"], how="left")
    result = result.merge(sig_feats, on=["date", "instrument"], how="left")

    # Step 8: Filter to signal date range and drop raw OHLCV columns
    result = result[result["primary_signal"].notna()].copy()
    raw_input_cols = {"open", "high", "low", "close", "volume", "open_interest"}
    meta_cols = ["date", "instrument", "primary_signal"]
    feat_cols = [c for c in result.columns if c not in meta_cols and c not in raw_input_cols]
    result = result[meta_cols + feat_cols]

    # Step 9: Sanity check for partial NaN post-warmup
    check_date = pd.Timestamp("2020-06-01")
    late = result[result["date"] >= check_date]
    for inst, g in late.groupby("instrument"):
        nan_rates = g[feat_cols].isna().mean()
        # Partially populated features (between 5% and 95% NaN) are flagged.
        # Fully NaN means the feature does not apply to this instrument (cross-class).
        partial_nan = nan_rates[(nan_rates > 0.05) & (nan_rates < 0.95)]
        if not partial_nan.empty:
            logger.warning(
                "Partial NaN (>5%%) for %s after %s: %s",
                inst, check_date.date(), partial_nan.index.tolist(),
            )

    # Step 10: Save
    feat_dict = _build_feature_dictionary()
    save_features(result, feat_dict, cfg)

    logger.info("=== Pipeline complete. Rows: %d, Feature cols: %d ===", len(result), len(feat_cols))
    return result


# Run the pipeline
# cfg = FeatureConfig()  # use defaults, or override paths here
# features = run_pipeline(cfg)
# features.head()

run_pipeline()


21:06:24 INFO === Feature Engineering Pipeline ===
21:06:24 INFO Loaded OHLCV: 65298 rows, 8 instruments, 1990-01-02 to 2022-06-30
21:06:24 INFO Computing base features...
21:06:31 INFO Base features done: 65298 rows, 40 columns
21:06:31 INFO Energy features done: 32684 rows, 16 columns
21:06:31 INFO Metals features done: 32696 rows, 13 columns
21:06:31 INFO Computing latent features...
21:06:31 INFO GMM refit at index 504 (date 1991-12-31), N_train=465
21:06:31 INFO GMM refit at index 567 (date 1992-03-31), N_train=528
21:06:31 INFO GMM refit at index 630 (date 1992-06-30), N_train=591
21:06:31 INFO GMM refit at index 693 (date 1992-09-29), N_train=654
21:06:31 INFO GMM refit at index 756 (date 1992-12-31), N_train=717
21:06:31 INFO GMM refit at index 819 (date 1993-04-01), N_train=780
21:06:32 INFO GMM refit at index 882 (date 1993-07-01), N_train=843
21:06:32 INFO GMM refit at index 945 (date 1993-09-30), N_train=906
21:06:32 INFO GMM refit at index 1008 (date 1994-01-03), N_train=9

,date,instrument,primary_signal,ret_1d,logret_1d,ret_5d,ret_10d,ret_20d,ret_60d,vol_20d,...,hmm_next_turbulent,hmm_regime_age,hmm_basket_turbulent_energy,hmm_basket_turbulent_metals,kmeans_cluster_id,kmeans_cluster_size,primary_signal_lag1,primary_signal_lag5,primary_signal_streak,primary_signal_agreement_5d
7543,2020-01-03,cl1s,0.0,0.030566,0.030108,0.021968,0.035516,0.077598,0.178520,0.009621,...,0.015823,54.0,0.998434,0.001323,2.0,0.75,NaN,NaN,0.0,NaN
7544,2020-01-06,cl1s,0.0,0.003489,0.003483,0.024803,0.033591,0.081081,0.182764,0.009578,...,0.011006,55.0,0.999299,0.001208,2.0,0.50,0.0,NaN,1.0,NaN
7545,2020-01-07,cl1s,-1.0,-0.009009,-0.009050,0.016402,0.036710,0.058939,0.155624,0.009757,...,0.010724,56.0,0.999323,0.002304,2.0,0.75,0.0,NaN,0.0,NaN
7546,2020-01-08,cl1s,0.0,-0.049282,-0.050538,-0.024034,-0.015151,0.011446,0.083838,0.015425,...,0.063378,57.0,0.985763,0.001970,2.0,1.00,-1.0,NaN,1.0,NaN
7547,2020-01-09,cl1s,0.0,-0.000839,-0.000839,-0.026836,-0.025691,0.006887,0.103500,0.015410,...,0.013175,58.0,0.998101,0.001626,2.0,1.00,0.0,NaN,2.0,-0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65293,2022-06-24,si1s,1.0,0.003944,0.003937,-0.035344,-0.032232,-0.034659,-0.160334,0.014966,...,0.020266,40.0,0.999921,0.999926,1.0,0.25,1.0,1.0,2.0,0.4
65294,2022-06-27,si1s,1.0,0.001323,0.001322,-0.020312,-0.036122,-0.037671,-0.174138,0.014917,...,0.018269,41.0,0.999909,0.999923,2.0,0.50,1.0,0.0,3.0,0.6
65295,2022-06-28,si1s,1.0,-0.014868,-0.014979,-0.043641,-0.019792,-0.058596,-0.189913,0.015072,...,0.022761,42.0,0.999930,0.999919,0.0,0.75,1.0,-1.0,4.0,1.0
65296,2022-06-29,si1s,1.0,-0.006420,-0.006441,-0.034012,-0.011970,-0.046400,-0.177111,0.014644,...,0.018582,43.0,0.999981,0.999814,0.0,0.75,1.0,1.0,5.0,1.0


In [31]:
pd.read_parquet('../features.parquet').head()

,date,instrument,primary_signal,ret_1d,logret_1d,ret_5d,ret_10d,ret_20d,ret_60d,vol_20d,...,hmm_next_turbulent,hmm_regime_age,hmm_basket_turbulent_energy,hmm_basket_turbulent_metals,kmeans_cluster_id,kmeans_cluster_size,primary_signal_lag1,primary_signal_lag5,primary_signal_streak,primary_signal_agreement_5d
0,2020-01-02,cl1s,0.0,0.001965,0.001963,0.001145,0.005080,0.088184,0.146135,0.011287,...,0.009877,53.0,0.999259,0.001567,0.0,0.75,NaN,NaN,0.0,NaN
1,2020-01-03,cl1s,1.0,0.030566,0.030108,0.021968,0.035516,0.077598,0.178520,0.009621,...,0.015823,54.0,0.998434,0.001323,0.0,0.75,0.0,NaN,0.0,NaN
2,2020-01-06,cl1s,1.0,0.003489,0.003483,0.024803,0.033591,0.081081,0.182764,0.009578,...,0.011006,55.0,0.999299,0.001208,0.0,0.50,1.0,NaN,1.0,NaN
3,2020-01-07,cl1s,0.0,-0.009009,-0.009050,0.016402,0.036710,0.058939,0.155624,0.009757,...,0.010724,56.0,0.999323,0.002304,0.0,0.75,1.0,NaN,2.0,NaN
4,2020-01-08,cl1s,-1.0,-0.049282,-0.050538,-0.024034,-0.015151,0.011446,0.083838,0.015425,...,0.063378,57.0,0.985763,0.001970,0.0,1.00,0.0,NaN,0.0,0.2
